In [0]:
from datetime import datetime, timedelta

from src.scripts.extract_data import ExtractData

In [0]:
def get_month_range(year, month):
    today = datetime.today()
    if today.year == year and today.month == month:
        date_start = datetime(year, month, 1)
        date_end = today
    else:
        date_start = datetime(year, month, 1)
        if month == 12:
            next_month = datetime(year + 1, 1, 1)
        else:
            next_month = datetime(year, month + 1, 1)
        date_end = next_month - timedelta(days=1)
    return date_start, date_end

In [0]:
dbutils.widgets.text("year", "")
dbutils.widgets.text("month", "")
dbutils.widgets.text("endpoint", "")

year_param = dbutils.widgets.get("year")
month_param = dbutils.widgets.get("month")
endpoint = dbutils.widgets.get("endpoint")

print(f"Parámetro year recibido: {year_param}")
print(f"Parámetro month recibido: {month_param}")
print(f"Parámetro endpoint recibido: {endpoint}")

current_date = datetime.now()
current_year = current_date.year
current_month = current_date.month

year = year_param if year_param else current_year
month = month_param if month_param else current_month
month_str = f"0{month}" if int(month) < 10 else str(month)
date_start, date_end = get_month_range(int(year), int(month))
period = f"{year}_{month_str}"
print({
    "date_start": date_start,
    "date_end": date_end,
    "period": period
})

In [0]:
%sql
SHOW TABLES IN formula_1;

In [0]:
extract_data = ExtractData()

In [0]:
meetings = extract_data.extract_meetings(date_start, date_end)

In [0]:
for dict in meetings:
    print(dict)
    meeting_key = dict.get("meeting_key", "")
    sessions = extract_data.extract_sessions(meeting_key, period)
    print(f"Se extrajeron {len(sessions)} sesiones para el meeting_key {meeting_key}")
    for session in sessions:
        session_key = session.get("session_key", "")
        drivers = extract_data.extract_drivers(meeting_key, session_key, period)
        print(f"Se extrajeron {len(drivers)} conductores para el meeting_key {meeting_key} y session_key {session_key}")
        for driver in drivers:
            driver_number = driver.get("driver_number", "")
            print(f"Se extrajeron datos para el driver_key {driver_number}")
            extract_data.extract_laps(meeting_key, session_key, driver_number, period)
            extract_data.extract_cars(session_key, driver_number, period)